### Installation

In [1]:
%%capture
!pip install unsloth  # Do this in local & cloud setups
!pip install langchain-text-splitters langchain-community pymupdf

In [2]:
import unsloth
from langchain_community.document_loaders import PyMuPDFLoader
import pandas as pd
from tqdm import tqdm
from transformers import TextStreamer
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)

==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-14b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2026.5.2 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [5]:
# 2025_url = 'https://eaa-online.org/app/uploads/sites/78/2026/02/program_v20250624.pdf'
# 2026_url = 'https://eaa-online.org/app/uploads/sites/80/2026/05/scientific_programme-open.pdf'

In [6]:
loader = PyMuPDFLoader('https://eaa-online.org/app/uploads/sites/78/2026/02/program_v20250624.pdf')
docs = loader.load()

data = pd.DataFrame()
for i in range(len(docs)):
    data.at[i, 'page'] = int(docs[i].metadata['page'])
    data.at[i, 'text'] = docs[i].page_content

len(data)

99

In [7]:
category_list = data['text'][0].split('. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .')[0]

In [8]:
system_prompt = {
    'name': 'system_prompt_v2', 'text':f'''
    Exctract category (you have to choose the category matching the abbreviation from the list of categories, e.g. FR - Financial Reporting, ED - Accounting Education), paper name, authors, and universities for every paper from the provided text.
    List of categories: {category_list}
    Output format:
    \n[Category]:[category];[Name]:[paper name];[Author_1]:[name of the 1st author];[Uni_1]:[university name of the 1st author];[Author_2]:[name of the 2nd author if exists];[Uni_2]:[university name of the 2nd author if exists];[Author_3]:[name of the 3rd author if exists];[Uni_3]:[university name of the 3rd author if exists];[Author_4]:[name of the 4th author if exists];[Uni_4]:[university name of the 4th author if exists];[Author_5]:[name of the 5th author if exists];[Uni_5]:[university name of the 5th author if exists]\n
'''
}

In [9]:
MAX_NEW_TOKENS = 2048
TEMPERATURE = 1
TOP_P = 0.95
TOP_K = 20

In [10]:
results = pd.DataFrame()
results['page'] = data['page']
results['text'] = data['text']

In [11]:
results['results'] = None
for i, ro in tqdm(results.iterrows(), total=len(data)):
    text = ro['text']

    # handling empty text
    if pd.isna(text) or not text.strip():
      continue

    messages = [{"role": "system", "content": system_prompt['text']},
                {"role": "user", "content": f"{text}"}]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # Must add for generation
        enable_thinking=False,        # Enable thinking
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    generated_ids = model.generate(
    **inputs,
    max_new_tokens = MAX_NEW_TOKENS,           # Increase for longer outputs!
    temperature = TEMPERATURE,
    top_p = TOP_P,
    top_k = TOP_K,
    do_sample = True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

    generated = tokenizer.decode(
    generated_ids[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
    )

    results.at[i, 'results'] = generated
results.to_csv(f'results_2025.csv')

  0%|          | 0/99 [00:00<?, ?it/s]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, F

[Category]:Financial Reporting;[Name]:Regulating by New Technology: The Impacts of the SEC Data Analytics on the SEC Investigations;[Author_1]:Tian Deng;[Uni_1]:CUHK-Shenzhen;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:Financial Reporting;[Name]:Do SPAC Combinations Affect Their Peers’ Financial Reporting Choices?;[Author_1]:Daniel Cohen;[Uni_1]:Vanderbilt University;[Author_2]:Sunay Mutlu;[Uni_2]:Kennesaw State University;[Author_3]:Kelly Ha;[Uni_3]:Kennesaw State University;[Author_4]:John Schomburger;[Uni_4]:Texas A&M University;[Author_5]:;[Uni_5]:  
[Category]:Financial Reporting;[Name]:Voluntary Public Disclosure of Revenues by Private Companies in the United States;[Author_1]:Thomas Bourveau;[Uni_1]:Columbia University;[Author_2]:Yiran Kang;[Uni_2]:City University of Hong Kong;[Author_3]:Wenqiang Pan;[Uni_3]:Columbia University;[Author_4]:Robert Stoumbos;[Uni_4]:ESSEC Business School;[Author_5]:;[Uni_5]:  
[Category]:Financial

  1%|          | 1/99 [01:20<2:11:46, 80.68s/it]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Category]:Financial Reporting;[Name]:Accounting conservatism and efficient project continuation revisited;[Author_1]:Jochen Bigus;[Uni_1]:Free University of Berlin  
[Category]:Financial Reporting;[Name]:Behavioral Conservatism;[Author_1]:Yasmin Hoffmann;[Uni_1]:University of Mannheim;[Author_2]:Qian Zhang;[Uni_2]:Tsinghua University;[Author_3]:Joaquin Peris Peris;[Uni_3]:Columbia University  
[Category]:Financial Reporting;[Name]:Reassessed Earnings with Capitalized Intangibles;[Author_1]:Anup Srivastava;[Uni_1]:University of Calgary;[Author_2]:Aneel Iqbal;[Uni_2]:Arizona State University;[Author_3]:Shiva Rajgopal;[Uni_3]:Columbia University;[Author_4]:Elnaz Basirianmahabadi;[Uni_4]:University of Calgary  
[Category]:Financial Reporting;[Name]:Board interlocks and corporate risk disclosures;[Author_1]:Amin Tavakkolnia;[Uni_1]:KU Leuven;[Author_2]:Dieter Smeulders;[Uni_2]:University of Bern  
[Category]:Financial Reporting;[Name]:Market Reactions to Mandatory Climate Disclosure: Evide

  2%|▏         | 2/99 [03:32<2:58:49, 110.61s/it]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

 98%|█████████▊| 97/99 [3:21:01<03:30, 105.46s/it]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Category]:History;[Name]:Accounting and inequality: An analysis of the treatise Arboleda de los Enfermos (ca. 1470);[Author_1]:Germán Gamero Igea;[Uni_1]:Universidad de Burgos;[Author_2]:Carlos Larrinaga;[Uni_2]:Universidad de Burgos;[Author_3]:María Uribe-Bohorquez;[Uni_3]:Pontificia Universidad Javeriana  
[Category]:History;[Name]:Before the advent of ERP, the questioning of double-entry bookkeeping in France between the 1960s and 1990s;[Author_1]:Yves Levant;[Uni_1]:Lille University;[Author_2]:Kada Meghraoui;[Uni_2]:University Paris Descartes;[Author_3]:Charles Ducrocq;[Uni_3]:University Paris Descartes  
[Category]:History;[Name]:Cultural legacies, Japanese corporations and their employees: Implications for Accounting;[Author_1]:Orie Miyazawa;[Uni_1]:University of Kent  
[Category]:History;[Name]:Balance Sheet Evolution of Gas Companies in London;[Author_1]:Chie Sawanobori;[Uni_1]:Osaka Sangyo University;[Author_2]:Mitsunori Kasukabe;[Uni_2]:Hokkaido University  
[Category]:Histo

 99%|█████████▉| 98/99 [3:22:57<01:48, 108.61s/it]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Category]:ISSB ResearchWorkshop;[Name]:Double Materiality as a Driver of Real Effects: Evidence from the European Union’s Non-Financial Disclosure Directive;[Author_1]:Peter Fiechter;[Uni_1]:University of Neuchatel;[Author_2]:Florian Habermann;[Uni_2]:University of Lausanne;[Author_3]:Gaia Melloni;[Uni_3]:HEC Lausanne / University of Lausanne;[Author_4]:Arianna Pisciella;[Uni_4]:Università Cattolica del Sacro Cuore  
[Category]:ISSB ResearchWorkshop;[Name]:The Effects of Human Capital Disclosures on Professional Investors’ Assessments of Firm Risk;[Author_1]:Ethan Rouen;[Uni_1]:Harvard Business School;[Author_2]:Lisa Laviers;[Uni_2]:Tulane University;[Author_3]:Jason Sandvik;[Uni_3]:University of Arizona;[Author_4]:Robert Jennings;[Uni_4]:University of Arizona<|im_end|>


100%|██████████| 99/99 [3:23:29<00:00, 123.32s/it]
